# Graph Analysis — JobGraph Knowledge Graph

This notebook examines the heterogeneous knowledge graph built by the graph builder.
We look at node/edge counts, degree distributions, centrality measures, connected
components, and visualize a company subgraph.

**Graph structure:** HeteroData with 3 node types (job, skill, company) and 3 edge types (requires, at, cooccurs).

**Key metrics from training:**
- Best val MRR: 0.486
- Test MRR: 0.290
- Test Hits@10: 62.7%

In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import networkx as nx
import torch

sns.set_theme(style="whitegrid")

GRAPH_DIR = Path("..") / "data" / "graph"
DATA_DIR = Path("..") / "data" / "processed"
print("Graph directory:", GRAPH_DIR.resolve())

## 1. Load Graph

In [ ]:
# Load the PyG HeteroData object
hetero_data = torch.load(GRAPH_DIR / "hetero_data.pt", weights_only=False)

# Load mappings (company names, skill names, job IDs)
with open(GRAPH_DIR / "mappings.json") as f:
    mappings = json.load(f)

# Also load the jobs dataframe for title lookups in visualizations
df = pd.read_parquet(DATA_DIR / "jobs.parquet")

print("=== Node Counts ===")
for node_type in hetero_data.node_types:
    n_nodes = hetero_data[node_type].x.shape[0]
    n_features = hetero_data[node_type].x.shape[1]
    print(f"  {node_type}: {n_nodes} nodes, {n_features} features")

print()
print("=== Edge Counts ===")
for edge_type in hetero_data.edge_types:
    n_edges = hetero_data[edge_type].edge_index.shape[1]
    print(f"  {edge_type}: {n_edges} edges")

print()
print(f"Companies: {mappings['companies']}")
print(f"Skills: {len(mappings['skills'])} unique")
print(f"Job IDs: {len(mappings['job_ids'])} total")

## 2. Degree Distributions

In [ ]:
def compute_degrees(edge_index, num_nodes, side=0):
    """Compute degree for each node from an edge_index tensor.

    Args:
        edge_index: [2, num_edges] tensor
        num_nodes:  total number of nodes on the chosen side
        side:       0 = source degrees, 1 = destination degrees
    """
    indices = edge_index[side].numpy()
    degrees = np.bincount(indices, minlength=num_nodes)
    return degrees


# Job degree = number of skills required
job_requires_skill = hetero_data["job", "requires", "skill"].edge_index
n_jobs = hetero_data["job"].x.shape[0]
job_degrees = compute_degrees(job_requires_skill, n_jobs, side=0)

# Skill degree = number of jobs requiring it
n_skills = hetero_data["skill"].x.shape[0]
skill_degrees = compute_degrees(job_requires_skill, n_skills, side=1)

# Company degree = number of jobs at the company
job_at_company = hetero_data["job", "at", "company"].edge_index
n_companies = hetero_data["company"].x.shape[0]
company_degrees = compute_degrees(job_at_company, n_companies, side=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(job_degrees, bins=30, color="#4ECDC4", edgecolor="white")
axes[0].set_title(f"Job Node Degree (skills required)\nMean={job_degrees.mean():.1f}")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Count")

axes[1].hist(skill_degrees, bins=30, color="#45B7D1", edgecolor="white")
axes[1].set_title(f"Skill Node Degree (jobs requiring it)\nMean={skill_degrees.mean():.1f}")
axes[1].set_xlabel("Degree")
axes[1].set_ylabel("Count")

axes[2].hist(company_degrees, bins=30, color="#FF6B6B", edgecolor="white")
axes[2].set_title(f"Company Node Degree (jobs posted)\nMean={company_degrees.mean():.1f}")
axes[2].set_xlabel("Degree")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 3. Centrality — Bridge Skills

In [ ]:
# Build a NetworkX graph of skill co-occurrence for centrality analysis
skill_cooccurs = hetero_data["skill", "cooccurs", "skill"].edge_index
skills_list = mappings["skills"]

G_skills = nx.Graph()
G_skills.add_nodes_from(range(len(skills_list)))

for i in range(skill_cooccurs.shape[1]):
    s1 = skill_cooccurs[0][i].item()
    s2 = skill_cooccurs[1][i].item()
    if s1 < s2:  # avoid duplicates (edges stored in both directions)
        G_skills.add_edge(s1, s2)

print(f"Skill co-occurrence graph: {G_skills.number_of_nodes()} nodes, {G_skills.number_of_edges()} edges")
print()

# Betweenness centrality — identifies skills that bridge different job clusters
betweenness = nx.betweenness_centrality(G_skills)
top_bridge_skills = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:20]

print("Top 20 Bridge Skills (Betweenness Centrality):")
print("-" * 45)
for idx, score in top_bridge_skills:
    name = skills_list[idx] if idx < len(skills_list) else f"Skill {idx}"
    print(f"  {name:25s} {score:.4f}")

In [ ]:
# Visualize top bridge skills
bridge_names = [skills_list[idx] for idx, _ in top_bridge_skills]
bridge_scores = [score for _, score in top_bridge_skills]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(bridge_names)), bridge_scores, color="#FF6B6B")
ax.set_yticks(range(len(bridge_names)))
ax.set_yticklabels(bridge_names)
ax.set_title("Top 20 Skills by Betweenness Centrality\n(Bridge skills connecting different job clusters)")
ax.set_xlabel("Betweenness Centrality")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Connected Components

In [ ]:
# Build a full heterogeneous graph in NetworkX for connectivity analysis
G_full = nx.Graph()

# Add all nodes with type prefixes to avoid ID collisions
for j in range(n_jobs):
    G_full.add_node(f"job_{j}", node_type="job")
for s in range(n_skills):
    G_full.add_node(f"skill_{s}", node_type="skill")
for c in range(n_companies):
    G_full.add_node(f"company_{c}", node_type="company")

# Add edges: job -> skill
for i in range(job_requires_skill.shape[1]):
    G_full.add_edge(f"job_{job_requires_skill[0][i].item()}",
                    f"skill_{job_requires_skill[1][i].item()}")

# Add edges: job -> company
for i in range(job_at_company.shape[1]):
    G_full.add_edge(f"job_{job_at_company[0][i].item()}",
                    f"company_{job_at_company[1][i].item()}")

# Add edges: skill -> skill (co-occurrence)
for i in range(skill_cooccurs.shape[1]):
    s1 = skill_cooccurs[0][i].item()
    s2 = skill_cooccurs[1][i].item()
    if s1 < s2:
        G_full.add_edge(f"skill_{s1}", f"skill_{s2}")

components = list(nx.connected_components(G_full))
component_sizes = sorted([len(c) for c in components], reverse=True)

print(f"Total nodes in full graph: {G_full.number_of_nodes()}")
print(f"Total edges in full graph: {G_full.number_of_edges()}")
print(f"Number of connected components: {len(components)}")
print()
print("Largest components (by node count):")
for i, size in enumerate(component_sizes[:10]):
    pct = 100 * size / G_full.number_of_nodes()
    print(f"  Component {i}: {size} nodes ({pct:.1f}%)")

# If there's one giant component, report graph density
if component_sizes[0] > 0.5 * G_full.number_of_nodes():
    giant = max(components, key=len)
    G_giant = G_full.subgraph(giant)
    print(f"\nGiant component density: {nx.density(G_giant):.6f}")
    print(f"Giant component avg clustering: {nx.average_clustering(G_giant):.4f}")

## 5. Subgraph Visualization

In [ ]:
# Build and visualize a company subgraph directly with networkx
# (self-contained — no project module imports needed)

companies_list = mappings["companies"]
skills_list = mappings["skills"]
job_ids_list = mappings["job_ids"]

# Pick the company with the most jobs for an interesting subgraph
company_job_counts = Counter()
for i in range(job_at_company.shape[1]):
    c_idx = job_at_company[1][i].item()
    if c_idx < len(companies_list):
        company_job_counts[companies_list[c_idx]] += 1

top_company = company_job_counts.most_common(1)[0][0]
top_company_idx = companies_list.index(top_company)
print(f"Visualizing subgraph for: {top_company} ({company_job_counts[top_company]} jobs)")

# Find jobs at this company (limit to 8 for readability)
MAX_JOBS = 8
job_mask = job_at_company[1] == top_company_idx
job_indices = job_at_company[0][job_mask].tolist()[:MAX_JOBS]

# Find skills required by those jobs
skill_indices = set()
job_skill_edges = []
for job_idx in job_indices:
    mask = job_requires_skill[0] == job_idx
    s_indices = job_requires_skill[1][mask].tolist()
    for s_idx in s_indices:
        skill_indices.add(s_idx)
        job_skill_edges.append((f"job_{job_idx}", f"skill_{s_idx}"))

# Find skill-skill co-occurrence edges within this subgraph
skill_cooccurs = hetero_data["skill", "cooccurs", "skill"].edge_index
skill_skill_edges = []
for i in range(skill_cooccurs.shape[1]):
    s1 = skill_cooccurs[0][i].item()
    s2 = skill_cooccurs[1][i].item()
    if s1 in skill_indices and s2 in skill_indices and s1 < s2:
        skill_skill_edges.append((f"skill_{s1}", f"skill_{s2}"))

# Build networkx graph
G_sub = nx.Graph()

# Company hub node
G_sub.add_node(top_company, node_type="company", label=top_company)

# Job nodes + edges to company
# Build a job_id -> title lookup from the dataframe
job_id_to_title = dict(zip(df["job_id"], df["title"]))
for j_idx in job_indices:
    jid = job_ids_list[j_idx] if j_idx < len(job_ids_list) else f"Job {j_idx}"
    title = job_id_to_title.get(jid, "")
    label = title[:30] if title else jid[:20]
    G_sub.add_node(f"job_{j_idx}", node_type="job", label=label)
    G_sub.add_edge(top_company, f"job_{j_idx}")

# Skill nodes
for s_idx in skill_indices:
    label = skills_list[s_idx] if s_idx < len(skills_list) else f"Skill {s_idx}"
    G_sub.add_node(f"skill_{s_idx}", node_type="skill", label=label)

# Job-skill and skill-skill edges
G_sub.add_edges_from(job_skill_edges)
G_sub.add_edges_from(skill_skill_edges)

print(f"Subgraph: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")
print(f"  Jobs: {len(job_indices)}, Skills: {len(skill_indices)}")

# Draw
NODE_COLOURS = {"company": "#FF6B6B", "job": "#4ECDC4", "skill": "#45B7D1"}
NODE_SIZES = {"company": 800, "job": 400, "skill": 200}

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(G_sub, k=2, iterations=50, seed=42)

for node_type, colour in NODE_COLOURS.items():
    nodes = [n for n, d in G_sub.nodes(data=True) if d.get("node_type") == node_type]
    if nodes:
        nx.draw_networkx_nodes(
            G_sub, pos, nodelist=nodes,
            node_color=colour, node_size=NODE_SIZES[node_type],
            alpha=0.9, ax=ax,
        )

nx.draw_networkx_edges(G_sub, pos, alpha=0.3, ax=ax)
labels = {n: d.get("label", n) for n, d in G_sub.nodes(data=True)}
nx.draw_networkx_labels(G_sub, pos, labels, font_size=7, ax=ax)

ax.set_title(f"JobGraph Subgraph: {top_company}", fontsize=14, fontweight="bold")
legend_handles = [Patch(color=c, label=t.title()) for t, c in NODE_COLOURS.items()]
ax.legend(handles=legend_handles, loc="upper left")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print("=== Graph Summary ===")
print(f"Jobs:      {n_jobs}")
print(f"Skills:    {n_skills}")
print(f"Companies: {n_companies}")
print(f"Job-Skill edges:     {job_requires_skill.shape[1]}")
print(f"Job-Company edges:   {job_at_company.shape[1]}")
print(f"Skill-Skill edges:   {skill_cooccurs.shape[1] // 2} (undirected)")
print(f"Connected components: {len(components)}")
print(f"Giant component:     {component_sizes[0]} nodes ({100 * component_sizes[0] / G_full.number_of_nodes():.1f}%)")
print(f"Top bridge skill:    {skills_list[top_bridge_skills[0][0]]}")
print()
print("Company job counts:")
for company, count in sorted(company_job_counts.items(), key=lambda x: -x[1]):
    print(f"  {company:25s} {count:4d} jobs")